# Sales Revenue Predictor

A machine learning pipeline to predict retail sales revenue from marketing and campaign features, covering data cleaning, preprocessing, model training, evaluation, and export.

## Import Libraries & Load Data

Imports pandas and loads the retail dataset from CSV into a DataFrame.

In [7]:
!wget https://raw.githubusercontent.com/AadityaKaushik/Sales-Revenue-Prediction/refs/heads/master/Retail_Data.csv

--2026-03-11 15:10:29--  https://raw.githubusercontent.com/AadityaKaushik/Sales-Revenue-Prediction/refs/heads/master/Retail_Data.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2689795 (2.6M) [text/plain]
Saving to: ‘Retail_Data.csv.1’

Retail_Data.csv.1   100%[===================>]   2.56M  --.-KB/s    in 0.04s   

2026-03-11 15:10:29 (65.4 MB/s) - ‘Retail_Data.csv.1’ saved [2689795/2689795]



In [8]:
import pandas as pd
data = pd.read_csv('Retail_Data.csv')

## Preview the Dataset

Displays the first 5 rows to verify the data loaded correctly and inspect column names and values.

In [9]:
data.head()

,id,date,region,channel,product_category,customer_segment,ad_spend,price,discount_rate,market_reach,impressions,click_through_rate,competition_index,seasonality_index,campaign_duration_days,customer_lifetime_value,sales_revenue
0,1,2011-12-05 11:31:00,Nort,Search,General,Standard,9.00,0.75,0.2782,32.0,817,0.0010,3.34,1.000000,30.0,816.49,119.767811
1,2,2011-04-27 14:01:00,North,Social Media,General,Premium,3.35,3.35,0.0912,61.0,2289,0.0640,4.44,0.366025,90.0,1723.16,119.404661
2,3,2010-11-09 15:20:00,North,Affiliate,General,Budget,2.55,2.55,0.1997,461.0,14697,0.1508,3.31,0.366025,21.0,1151.74,132.009747
3,4,2010-10-03 15:24:00,North,Affiliate,Storage,Premium,2.95,2.95,0.4767,744.0,17578,0.1965,2.87,-0.366025,90.0,3585.85,154.511756
4,5,2011-10-14 09:28:00,North,Search,Lighting,Premium,15.00,1.25,0.3536,226.0,6280,0.0200,7.40,-0.366025,90.0,502.28,128.059924


## Clean the Region Column

Standardises the `region` column by lowercasing, stripping whitespace, and correcting known typos like `'Nort'` and `'norht'`.

In [10]:
data['region'].unique()
data['region'] = (
    data['region']
    .str.strip()
    .str.lower()
    .replace({
        'norht': 'north',
        'nort': 'north'
    })
)

## Check for Missing Values

Counts `NaN` values per column to understand the extent of missingness before preprocessing.

In [11]:
data.isnull().sum()

,0
id,0
date,0
region,0
channel,0
product_category,0
customer_segment,0
ad_spend,658
price,0
discount_rate,755
market_reach,686


## Inspect Data Types & Shape

Prints column types, non-null counts, and total row count to confirm the dataset structure.

In [12]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18000 entries, 0 to 17999
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       18000 non-null  int64  
 1   date                     18000 non-null  object 
 2   region                   18000 non-null  object 
 3   channel                  18000 non-null  object 
 4   product_category         18000 non-null  object 
 5   customer_segment         18000 non-null  object 
 6   ad_spend                 17342 non-null  float64
 7   price                    18000 non-null  float64
 8   discount_rate            17245 non-null  float64
 9   market_reach             17314 non-null  float64
 10  impressions              18000 non-null  int64  
 11  click_through_rate       17287 non-null  float64
 12  competition_index        17293 non-null  float64
 13  seasonality_index        18000 non-null  float64
 14  campaign_duration_days

## Verify Numeric Compatibility

Confirms that `campaign_duration_days` and `market_reach` can be safely cast to float, as required for imputation.

In [13]:
data[['campaign_duration_days', 'market_reach']].astype('float')

,campaign_duration_days,market_reach
0,30.0,32.0
1,90.0,61.0
2,21.0,461.0
3,90.0,744.0
4,90.0,226.0
...,...,...
17995,14.0,1115.0
17996,30.0,3.0
17997,7.0,174.0
17998,7.0,227.0


## Define Features & Split the Data

Separates the target (`sales_revenue`) from the features, dropping `date` and `id`, then splits into 80% train and 20% test sets.

In [14]:
from sklearn.model_selection import train_test_split
y = data['sales_revenue']
X = data.drop(['sales_revenue', 'date', 'id'], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, train_size=0.8, shuffle=True)
X_train


,region,channel,product_category,customer_segment,ad_spend,price,discount_rate,market_reach,impressions,click_through_rate,competition_index,seasonality_index,campaign_duration_days,customer_lifetime_value
16210,north,Search,Kitchen,Premium,75.60,2.10,NaN,87.0,1747,0.0381,2.61,1.366025,21.0,5239.501
9259,north,Search,General,Premium,NaN,1.25,NaN,82.0,1086,0.0357,4.54,0.366025,60.0,9660.130
7677,north,Email,Storage,Premium,13.20,0.55,0.1035,112.0,2022,0.0284,3.01,1.000000,NaN,2276.860
10145,north,Affiliate,General,Premium,11.90,5.95,0.0561,321.0,4434,0.0374,6.69,0.366025,7.0,326.400
1543,north,Social Media,Lighting,Premium,27.00,6.75,0.2367,174.0,6478,0.0232,1.84,0.366025,60.0,448.510
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12321,north,Social Media,General,Premium,8.25,8.25,0.2044,251.0,7512,0.0188,3.23,-0.366025,14.0,6706.360
9859,north,Influencer,Seasonal,Standard,3.40,0.85,0.1992,180.0,7543,0.0168,7.60,1.366025,7.0,267.880
4924,north,Social Media,Storage,Premium,45.00,1.25,0.0274,57.0,783,0.0694,1.95,0.366025,21.0,53405.670
12365,north,Influencer,Seasonal,Budget,8.25,1.65,0.4636,207.0,2892,0.0793,2.72,0.366025,60.0,61.100


## Build the Preprocessing Pipeline

Defines a `ColumnTransformer` that imputes and scales numeric columns, and one-hot encodes categorical ones. The final `pipeline1` appends a **Ridge regression** model.

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

num_cols = [
    'ad_spend',
    'price',
    'discount_rate',
    'market_reach',
    'impressions',
    'click_through_rate',
    'competition_index',
    'seasonality_index',
    'campaign_duration_days',
    'customer_lifetime_value'
]
cat_cols = [
    'region',
    'channel',
    'product_category',
    'customer_segment'
]

num_pipeline = Pipeline([('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),])

cat_pipeline = Pipeline([('encoder', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

pipeline1 = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge())
])

## Train the Ridge Regression Model

Fits the full pipeline on the training data — preprocessing and model training happen in a single call.

In [16]:
pipeline1.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['ad_spend', 'price',
                                                   'discount_rate',
                                                   'market_reach',
                                                   'impressions',
                                                   'click_through_rate',
                                                   'competition_index',
                                                   'seasonality_index',
                                                   'campaign_duration_days',
                                                   'customer_lifetime_value']),
                                                 ('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['region', 'channel',
                                                   'product_category',
                                                   'customer_segment'])])),
                ('model', Ridge())])

## Evaluate the Ridge Regression Model

Runs predictions on the test set and reports MAE, MSE, R², and MAPE to assess model performance.

In [17]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
y_pred = pipeline1.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae}")

mse = mean_squared_error(y_test, y_pred)
print(f"MSE: {mse}")

r2 = r2_score(y_test, y_pred)
print(f"R squared: {r2}")

import numpy as np
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
print("MAPE:", mape)

MAE: 32.62199522758489
MSE: 2039.854756757528
R squared: 0.3242669207184886
MAPE: 26.38310720592231


## Tune a Second Ridge Model with Cross-Validation

Uses `GridSearchCV` with 5-fold CV to find the best `alpha` for a second Ridge pipeline, searching across [0.01, 0.1, 1, 10, 100].

In [18]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

pipeline2 = Pipeline([('preprocessor', preprocessor), ('model', Ridge())])
param_grid = {
    "model__alpha": [0.01, 0.1, 1, 10, 100]
}
grid = GridSearchCV(pipeline2, param_grid, cv=5, scoring="neg_mean_absolute_error")
grid.fit(X_train, y_train)

print("Best alpha:", grid.best_params_)
print("Best CV MAE:", -grid.best_score_)

Best alpha: {'model__alpha': 10}
Best CV MAE: 32.535775369279015


## Evaluate the Best Tuned Model

Scores the best pipeline from grid search on the test set to compare against the baseline Ridge model.

In [19]:
best_pipeline = grid.best_estimator_
y_pred2 = best_pipeline.predict(X_test)
mae = mean_absolute_error(y_test, y_pred2)
mse = mean_squared_error(y_test, y_pred2)
r2 = r2_score(y_test, y_pred2)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"R squared: {r2}")
print("MAPE:", mape)

MAE: 32.62006586672139
MSE: 2039.5626784176525
R squared: 0.3243636761346974
MAPE: 26.38310720592231


## Save the Trained Pipeline

Serialises the best pipeline to a `.pkl` file using `joblib` for later use without retraining.

In [20]:
import joblib

joblib.dump(best_pipeline, "Sales_Revenue_Pipeline.pkl")
print("Pipeline successfully saved!")

Pipeline successfully saved!
